## Strategy

---

> **In one line.** A family of interchangeable algorithms shares one type signature, and a context object holds a *slot* for exactly one of them, delegating all work to whichever currently occupies it — so swapping the occupant changes behavior without touching the context.

### 1. The family and its shared signature

Fix an **input domain** $X$ (the type of data every strategy consumes, e.g. a list to be sorted) and an **output codomain** $Y$ (the type of result every strategy produces, e.g. the sorted list). A **strategy** is a function $f_i : X \rightarrow Y$ — one concrete realization of an algorithm, such as bubble sort, merge sort, or Python's built-in sort. The pattern collects all such realizations into a single set, the **strategy family**

$$\mathcal{F} = \{f_1, f_2, \ldots, f_n\}, \qquad \text{where each } f_i : X \rightarrow Y.$$

The decisive fact is that *every* member carries the **same** type $X \rightarrow Y$. The functions differ in their internals, never in their interface. It is precisely this shared signature that makes the $f_i$ mutually substitutable: any one may stand wherever any other could.

### 2. The context as a delegating slot

A **`Context`** is an object carrying a single slot that holds one member $f_i \in \mathcal{F}$. The context contributes no algorithm of its own; when invoked on an input $x \in X$ it simply *forwards* the call to whatever occupies the slot:

$$\boxed{\,\text{Context}(f_i)(x) \;=\; f_i(x)\,}$$

Read left to right, this is pure delegation: parametrize the context by a strategy, hand it an input, and it returns exactly what that strategy returns. Data flows straight through the slot:

$$\underbrace{x}_{X} \;\xrightarrow{\;\text{slot}\,=\,f_i\;}\; \underbrace{\text{Context}(f_i)(x)}_{=\,f_i(x)} \;=\; \underbrace{f_i(x)}_{Y}.$$

Because the slot is the only moving part, selecting an algorithm is the act of choosing which $f_i$ to drop into it:

$$\text{client} \xrightarrow{\;\text{set strategy}\;} \text{Context}[\,f_i\,] \xrightarrow{\;\text{delegate}\;} f_i \in \mathcal{F}.$$

### 3. Conditions

1. **Shared signature** — every $f_i$ has the identical type $X \rightarrow Y$. This common interface is what makes the members interchangeable in the context's slot.
   $$\forall\, f_i \in \mathcal{F} : \quad f_i : X \rightarrow Y$$
2. **Runtime swap** — replacing the occupant $f_i$ with another member $f_j$ is the *entire* mechanism for changing the algorithm; the context itself is never altered.
   $$\text{slot} : f_i \;\longmapsto\; f_j, \qquad \text{Context unchanged}$$
3. **Open–closed** — a new strategy $f_{n+1}$ may be admitted into $\mathcal{F}$ without modifying the context, which knows only the abstract type $X \rightarrow Y$.
   $$\mathcal{F} \;\longmapsto\; \mathcal{F} \cup \{f_{n+1}\}, \qquad f_{n+1} : X \rightarrow Y$$

> 🗺️ A navigation app. The destination ($X$) and the returned route ($Y$) are fixed types. The route-finding algorithm ($f_i$) — driving, cycling, walking — swaps out entirely. The app's map (the context) never changes.

### Exercise 03 — Sorting Strategy

---

**Scenario:** A `DataSorter` (context) should sort lists. The algorithm ($f_i \in \mathcal{F}$) is swappable at runtime — bubble sort, merge sort, built-in sort.

**Your task:** Build `DataSorter` and three strategy classes. The sorter delegates entirely to whichever $f_i$ is in its slot.

```python
sorter = DataSorter(strategy=BubbleSortStrategy())   # slot <- f_1
sorter.sort([3, 1, 4])
sorter.set_strategy(MergeSortStrategy())             # slot <- f_2
sorter.sort([3, 1, 4])                               # same call, different f_i
```

**Hints**

- The context stores `self._strategy = f_i`. Its `sort(data)` calls `self._strategy.sort(data)` — full delegation to the slot.
- All strategy classes implement the same method name `sort(data)` — this is the shared type $X \rightarrow Y$ that makes them interchangeable.

In [ ]:
from abc import ABC, abstractmethod

# --------------------------------
# Shared signature X -> Y: every strategy implements sort(data)

class SortStrategy(ABC):
    @abstractmethod
    def sort(self, data): ...                # f_i : X -> Y

# --------------------------------
# Concrete strategies (members of F) — each implements sort(data)

class BubbleSortStrategy(SortStrategy):       # f_1
    def sort(self, data):
        # your task: return a sorted copy via bubble sort (X -> Y)
        ...

class MergeSortStrategy(SortStrategy):        # f_2
    def sort(self, data):
        # your task: return a sorted copy via merge sort (X -> Y)
        ...

class BuiltinSortStrategy(SortStrategy):      # f_3
    def sort(self, data):
        # your task: return a sorted copy using Python's sorted() (X -> Y)
        ...

# --------------------------------
# Context: holds one f_i in its slot and delegates entirely to it

class DataSorter:
    def __init__(self, strategy):
        self._strategy = strategy            # slot <- f_i

    def set_strategy(self, strategy):        # runtime swap: slot <- f_j
        self._strategy = strategy

    def sort(self, data):                    # Context(f_i)(x) = f_i(x)
        # your task: delegate to self._strategy.sort(data)
        ...

# --------------------------------
sorter = DataSorter(strategy=BubbleSortStrategy())   # slot <- f_1
print(sorter.sort([3, 1, 4]))
sorter.set_strategy(MergeSortStrategy())             # slot <- f_2
print(sorter.sort([3, 1, 4]))                        # same call, different f_i

### Exercise 04 — Discount Strategy

---

**Scenario:** A shopping cart applies discounts. $\mathcal{F}$ = {NoDiscount, PercentOff, FixedOff, BOGO}. The cart (context) delegates all discount logic to whichever $f_i$ is active.

**Your task:** Build the cart context and four strategies. Each strategy implements `apply(price)` → float — the shared signature $X \rightarrow Y$.

```python
cart = Cart(discount=NoDiscount())     # slot <- f_1
cart.checkout(100.0)
cart.set_discount(PercentOff(20))      # slot <- f_2
cart.checkout(100.0)                   # same call, different f_i
```

**Hints**

- The cart never contains an if/elif chain for discount types. It only calls `self._discount.apply(total)`. All conditional logic lives inside the strategy objects in $\mathcal{F}$.
- Every strategy implements the same `apply(price) -> float` — the shared signature $X \rightarrow Y$ that makes them interchangeable in the cart's slot.

In [ ]:
from abc import ABC, abstractmethod

# --------------------------------
# Shared signature X -> Y: every strategy implements apply(price) -> float

class DiscountStrategy(ABC):
    @abstractmethod
    def apply(self, price): ...              # f_i : X -> Y

# --------------------------------
# Concrete strategies (members of F) — all conditional logic lives here

class NoDiscount(DiscountStrategy):           # f_1
    def apply(self, price):
        # your task: return price unchanged (X -> Y)
        ...

class PercentOff(DiscountStrategy):           # f_2
    def __init__(self, percent):
        self._percent = percent
    def apply(self, price):
        # your task: return price reduced by self._percent% (X -> Y)
        ...

class FixedOff(DiscountStrategy):             # f_3
    def __init__(self, amount):
        self._amount = amount
    def apply(self, price):
        # your task: return price minus self._amount, not below 0 (X -> Y)
        ...

class BOGO(DiscountStrategy):                 # f_4 (buy-one-get-one: half price)
    def apply(self, price):
        # your task: return half of price (X -> Y)
        ...

# --------------------------------
# Context: holds one f_i in its slot and delegates entirely to it

class Cart:
    def __init__(self, discount):
        self._discount = discount            # slot <- f_i

    def set_discount(self, discount):        # runtime swap: slot <- f_j
        self._discount = discount

    def checkout(self, total):               # Context(f_i)(x) = f_i(x)
        # your task: delegate to self._discount.apply(total) — no if/elif here
        ...

# --------------------------------
cart = Cart(discount=NoDiscount())           # slot <- f_1
print(cart.checkout(100.0))
cart.set_discount(PercentOff(20))            # slot <- f_2
print(cart.checkout(100.0))                  # same call, different f_i